# Download fastMRI knee single-coil data directly to Google Drive

This notebook streams the official archives into mounted Google Drive. It does not store a full copy on the Colab runtime or your local HDD.

Before starting:

1. Add the signed URLs under Colab **Secrets** (key icon) as `FASTMRI_KNEE_VAL_URL` and `FASTMRI_KNEE_TRAIN_URL`.
2. Enable notebook access for both secrets.
3. Ensure Drive has enough space for the archives and extracted HDF5 data.
4. Download validation first; it is smaller and sufficient for loader development.

Signed URLs are never printed or written into this notebook.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/fastMRI')
ARCHIVE_DIR = DRIVE_ROOT / 'archives'
RAW_DIR = DRIVE_ROOT / 'raw'
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

print('[setup] archive directory:', ARCHIVE_DIR)
print('[setup] extraction directory:', RAW_DIR)

def ensure_pv():
    if shutil.which('pv') is None:
        print('[setup] installing pv for archive progress')
        subprocess.run(['apt-get', 'update', '-qq'], check=True)
        subprocess.run(['apt-get', 'install', '-y', '-qq', 'pv'], check=True)


def run_tar_with_progress(archive, tar_arguments, quiet=False):
    ensure_pv()
    archive_size = archive.stat().st_size
    pv_process = subprocess.Popen(
        [
            'pv', '--progress', '--timer', '--eta', '--rate', '--bytes',
            '--size', str(archive_size), str(archive),
        ],
        stdout=subprocess.PIPE,
    )
    try:
        subprocess.run(
            ['tar', *tar_arguments],
            stdin=pv_process.stdout,
            stdout=subprocess.DEVNULL if quiet else None,
            check=True,
        )
    finally:
        if pv_process.stdout is not None:
            pv_process.stdout.close()
    pv_returncode = pv_process.wait()
    if pv_returncode != 0:
        raise subprocess.CalledProcessError(pv_returncode, pv_process.args)

## Download archives

Downloads are disabled by default. `curl --continue-at -` resumes an existing partial archive after a Colab disconnect. The progress bar reports bytes streamed directly into mounted Drive, including speed and ETA. Start with validation, then enable training.

In [ ]:
RUN_DOWNLOADS = False
DOWNLOAD_VALIDATION = True
DOWNLOAD_TRAINING = False

DOWNLOADS = {
    'knee_singlecoil_val.tar.xz': 'FASTMRI_KNEE_VAL_URL',
    'knee_singlecoil_train.tar.xz': 'FASTMRI_KNEE_TRAIN_URL',
}
selected = []
if DOWNLOAD_VALIDATION:
    selected.append('knee_singlecoil_val.tar.xz')
if DOWNLOAD_TRAINING:
    selected.append('knee_singlecoil_train.tar.xz')

if RUN_DOWNLOADS:
    for filename in selected:
        destination = ARCHIVE_DIR / filename
        secret_name = DOWNLOADS[filename]
        url = userdata.get(secret_name)
        print(f'[download] starting/resuming {filename}')
        subprocess.run(
            [
                'curl',
                '--fail',
                '--location',
                '--progress-bar',
                '--show-error',
                '--continue-at',
                '-',
                '--retry',
                '8',
                '--retry-all-errors',
                '--output',
                str(destination),
                url,
            ],
            check=True,
        )
        size_gib = destination.stat().st_size / 1024**3
        print(f'[download] complete {filename} ({size_gib:.2f} GiB)')
else:
    print('[download] disabled; set RUN_DOWNLOADS = True when ready')

## Validate archive integrity

This reads the archive index without extracting it. The progress display reports compressed bytes read from Drive. Run it after each completed download.

In [ ]:
RUN_ARCHIVE_CHECKS = False

if RUN_ARCHIVE_CHECKS:
    archives = sorted(ARCHIVE_DIR.glob('knee_singlecoil_*.tar.xz'))
    if not archives:
        raise FileNotFoundError(f'no fastMRI archives found in {ARCHIVE_DIR}')
    for archive in archives:
        print(f'[verify] checking {archive.name}')
        run_tar_with_progress(archive, ['-tJf', '-'], quiet=True)
        print(f'[verify] archive is readable: {archive.name}')
else:
    print('[verify] disabled; set RUN_ARCHIVE_CHECKS = True after download')

## Extract into Google Drive

Extraction reads and writes through Drive. The progress display reports compressed bytes read, speed, and ETA while `tar` writes the extracted files to Drive. It may be slow, but no complete local copy is created. Keep the archive until HDF5 verification succeeds.

In [ ]:
RUN_EXTRACTION = False
EXTRACT_VALIDATION = True
EXTRACT_TRAINING = False

to_extract = []
if EXTRACT_VALIDATION:
    to_extract.append(ARCHIVE_DIR / 'knee_singlecoil_val.tar.xz')
if EXTRACT_TRAINING:
    to_extract.append(ARCHIVE_DIR / 'knee_singlecoil_train.tar.xz')

if RUN_EXTRACTION:
    for archive in to_extract:
        if not archive.is_file():
            raise FileNotFoundError(archive)
        print(f'[extract] {archive.name}')
        run_tar_with_progress(archive, ['-xJf', '-', '-C', str(RAW_DIR)])
        print(f'[extract] complete: {archive.name}')
else:
    print('[extract] disabled; verify archives before enabling extraction')

## Verify extracted HDF5 volumes

The cell reports counts, k-space shapes, and available keys without loading full volumes into memory. Enable the full check to validate every volume with a progress bar.

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', 'h5py>=3.9', 'tqdm>=4.66'],
    check=True,
)

import h5py
from tqdm.auto import tqdm

RUN_FULL_HDF5_CHECK = False

h5_files = sorted(RAW_DIR.rglob('*.h5'))
print(f'[hdf5] found {len(h5_files):,} volumes')

if h5_files:
    sample = h5_files[0]
    with h5py.File(sample, 'r') as handle:
        print('[hdf5] sample:', sample.relative_to(RAW_DIR))
        print('[hdf5] keys:', sorted(handle.keys()))
        print('[hdf5] kspace:', handle['kspace'].shape, handle['kspace'].dtype)
        if 'reconstruction_esc' in handle:
            print('[hdf5] reconstruction_esc:', handle['reconstruction_esc'].shape)

    if RUN_FULL_HDF5_CHECK:
        print('[hdf5] validating every volume')
        for path in tqdm(h5_files, desc='HDF5 volumes', unit='volume'):
            with h5py.File(path, 'r') as handle:
                if 'kspace' not in handle or handle['kspace'].ndim != 3:
                    raise ValueError(f'invalid single-coil k-space in {path}')
        print(f'[hdf5] validated {len(h5_files):,} volumes')
    else:
        print('[hdf5] full check disabled; set RUN_FULL_HDF5_CHECK = True to run it')
else:
    print('[hdf5] no extracted volumes found yet')

## Optional archive cleanup

Only delete archives after the HDF5 count and sample inspection are correct. This action cannot be resumed afterward without downloading again.

In [ ]:
CONFIRM_ARCHIVE_DELETION = ''

if CONFIRM_ARCHIVE_DELETION == 'DELETE VERIFIED ARCHIVES':
    if not h5_files:
        raise RuntimeError('refusing cleanup because no extracted HDF5 files were verified')
    for archive in sorted(ARCHIVE_DIR.glob('knee_singlecoil_*.tar.xz')):
        print(f'[cleanup] deleting {archive.name}')
        archive.unlink()
    print('[cleanup] complete')
else:
    print('[cleanup] disabled')